Check before running:
1. Check api key is correctly named and stored
2. check the model name
3. Check the TEMPERATURE
4. Check max tokens
5. Check the iterations
6. Paste the description
7. Change the ID of description
8. Rerun only the second part of the code

In [ ]:
# experiment_runner_gemini_zero_shot.py

# ── 0. Setup ───────────────────────────────────────────────
!pip install -q google-genai pandas

from google import genai
from google.genai import types
from google.colab import files
from google.colab import userdata
import pandas as pd
import requests
import os
import re

os.makedirs("experiments/dmn", exist_ok=True)

In [ ]:
# ── 1. API key and model ───────────────────────────────────
API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

#Stick to one model but check which is better
MODEL_NAME = "gemini-3-flash-preview"

In [ ]:
import requests
from lxml import etree
import os # Added import for os module

# ── 2. Configuration ─────────────────
# Change manually per run: 0.2 / 0.4 / 0.6
TEMPERATURE = 0.2

N_ITERATIONS = 3
MAX_TOKENS = 35000
TOP_P = 1.0

# ── DMN Schema and Validation Setup (Moved from cell 0) ─────────────────
# DMN 1.3 Schema URL - Using a common OMG link that works
# ── DMN Schema and Validation Setup ─────────────────
# ── DMN Schema and Validation Setup ─────────────────
DMN_SCHEMA_FILE = "DMN13.xsd"

SCHEMA_FILES = {
    "DMN13.xsd":   "https://www.omg.org/spec/DMN/20191111/DMN13.xsd",
    "DMNDI13.xsd": "https://www.omg.org/spec/DMN/20191111/DMNDI13.xsd",
    "DC.xsd":      "https://www.omg.org/spec/DMN/20180521/DC.xsd",
    "DI.xsd":      "https://www.omg.org/spec/DMN/20180521/DI.xsd",
}

xmlschema = None

for filename, url in SCHEMA_FILES.items():
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        try:
            r = requests.get(url)
            r.raise_for_status()
            with open(filename, "wb") as f:
                f.write(r.content)
            print(f"  Saved {filename}")
        except requests.exceptions.RequestException as e:
            print(f"  Error downloading {filename}: {e}")

try:
    xmlschema_doc = etree.parse(DMN_SCHEMA_FILE)
    xmlschema = etree.XMLSchema(xmlschema_doc)
    print("DMN schema parsed successfully.")
except Exception as e:
    print(f"Error parsing DMN schema: {e}. Falling back to basic XML validation.")
    xmlschema = None

def validate_dmn_xml(xml_string: str) -> (bool, list):
    """
    Validates a DMN XML string against the pre-loaded DMN schema, or performs basic
    XML syntax validation if the DMN schema could not be loaded.

    Args:
        xml_string: The DMN XML content as a string.

    Returns:
        A tuple: (is_valid, errors_list). is_valid is True if valid, False otherwise.
        errors_list contains validation errors if any.
    """
    try:
        # First, attempt basic XML parsing to catch malformed XML quickly
        xml_doc = etree.fromstring(xml_string.encode('utf-8'))

        # If DMN schema was loaded, perform full XSD validation
        if xmlschema is not None:
            try:
                xmlschema.assertValid(xml_doc)
                return True, []
            except etree.XMLSchemaError as e:
                return False, [f"DMN Schema Validation Error: {e}"]
        else:
            # If DMN schema could not be loaded, report basic XML validity
            return True, ["Warning: DMN schema not loaded. Performed basic XML syntax check only."]

    except etree.XMLSyntaxError as e:
        print(f"  [validation] XML Syntax Error: {e}")
        print(f"  [validation] Malformed XML (from validate_dmn_xml):\n{xml_string[:500]}...\n") # Print part of it
        return False, [f"XML Syntax Error: {e}"]
    except Exception as e:
        return False, [f"An unexpected error occurred during XML validation: {e}"]

# ── 3. New description ─────────────────
description_id = "description_id"

description = """enter your description here"""

# ── 4. Zero-shot prompt ─────────────────
def build_zero_shot_prompt(description: str) -> str:
    return f"""

    You are an expert in generating DMN 1.3 XML compatible with Camunda. (Persona)

Generate a complete and executable DMN XML file from the textual description below. (Constraint Instruction)

STRICT OUTPUT RULES:
- Output ONLY XML.
- No explanations, comments, or markdown.
- The XML must start with: <?xml version=\"1.0\" encoding=\"UTF-8\"?>
- The XML must end with </definitions>.
- All tags must be properly closed.

NAMESPACE RULES:
- Use default namespace:
  xmlns=\"https://www.omg.org/spec/DMN/20191111/MODEL/\"
- Include the following namespaces for diagram interchange:
  xmlns:dmndi=\"https://www.omg.org/spec/DMN/20191111/DMNDI/\"
  xmlns:dc=\"http://www.omg.org/spec/DMN/20180521/DC/\"
  xmlns:modeler=\"http://camunda.org/schema/modeler/1.0\"
  xmlns:biodi=\"http://bpmn.io/schema/dmn/biodi/2.0\"
  xmlns:di=\"http://www.omg.org/spec/DMN/20180521/DI/\"

STRUCTURE RULES:
- The root element must be:
  <definitions id=\"Definitions_00ghp5h\" name=\"DRD\" namespace=\"http://example.com/dmn\" exporter=\"Camunda Modeler\" exporterVersion=\"5.44.0\" modeler:executionPlatform=\"Camunda Cloud\" modeler:executionPlatformVersion=\"8.8.0\">

- Include inputData elements when variables are present.
- Each inputData must contain:
  <variable name=\"VARIABLE_NAME\" typeRef=\"VARIABLE_TYPE\"/>

- Include at least one decision:
  <decision id=\"DECISION_ID\" name=\"DECISION_NAME\">

- Each decision must contain exactly one decisionTable.

DECISION TABLE RULES:
- decisionTable must include:
  - one or more <input>
  - exactly one <output name=\"OUTPUT_NAME\" typeRef=\"OUTPUT_TYPE\"/>
  - one or more <rule>
- decisionTable MUST have hitPolicy=\"UNIQUE\":
  <decisionTable id=\"decisionTable_ID\" hitPolicy=\"UNIQUE\">

- Each input must contain:
  <inputExpression id=\"INPUT_EXPRESSION_ID\" typeRef=\"INPUT_TYPE\">
    <text>variable_name</text>
  </inputExpression>

- Each rule must contain:
  - one <inputEntry> per input
  - one <outputEntry>

- inputEntry format:
  <inputEntry><text>FEEL_condition</text></inputEntry>

- Use "-" inside <inputEntry><text>-</text></inputEntry> to represent any value ("don't care").
- Apply this convention consistently in all decision table rules.

- outputEntry format:
  <outputEntry><text>FEEL_value</text></outputEntry>

FEEL RULES:
- Use FEEL unary tests:
  - exact match: "sunny"
  - less than: < 18
  - greater or equal than:>= 18
  - half-open interval: [5..10[   ← includes 18, excludes 65
  - half-open interval: ]2..5]    ← excludes 18, includes 65
  - closed interval: [18..65]
- Strings must be in double quotes
- Do not use programming operators like ==, &&, ||

ID RULES:
- All ids must be unique
- Use consistent naming:
  - inputData: input_<name>
  - decision: decision_<name>
  - rules: rule_<number>

QUALITY RULES:
- Ensure the XML is executable in Camunda
- Ensure logic matches the description
- No placeholders or incomplete elements

========================
LAYOUT SYSTEM (CRITICAL)
========================

GRID RULES:
- Row 1 (decision): y = 100
- Row 2 (inputs):   y = 300

INPUT LAYOUT RULES:
- All input nodes on the SAME horizontal row (y = 300)
- x = 100 + (index * 200), index starts at 0
- width = 160, height = 60

DECISION LAYOUT RULES:
- Decision centered above all inputs
- x = midpoint of all input x-coordinates
- y = 100
- width = 180, height = 80

WAYPOINT RULES:
- Edge source = top-center of input shape:
    x = input_x + 80
    y = input_y  ← top edge of input (y=300, NOT y+height)
- Edge target = ALWAYS the same point for ALL edges = bottom-center of decision:
    x = decision_x + 90
    y = decision_y + 80  (bottom edge, since decision height=80)

CRITICAL: Every single DMNEdge must have the IDENTICAL target waypoint.   ← CRITICAL block
All edges share one convergence point at the bottom-center of the decision node.
NEVER use decision_x alone as the target x — always add half the width.
EXAMPLE for decision at x=400, width=180, height=80:
  target waypoint → x=490, y=180

EXAMPLE for input at x=100, width=160:
  source waypoint → x=180, y=300

INPUTDATA LABEL RULES:
- Every <inputData> MUST have a name attribute matching its variable:
  <inputData id=\"input_allergies\" name=\"allergies\">
- NEVER leave name empty

========================
DMNDI RULES:
========================

LOGICAL STRUCTURE (must come BEFORE dmndi section):
- Each <decision> that depends on an inputData must contain:
  <informationRequirement id=\"IR_<decisionId>_<inputDataId>\">
    <requiredInput href=\"#<inputDataId>\"/>
  </informationRequirement>

- Give EVERY informationRequirement a UNIQUE id like \"IR_decision1_input_allergies\"

DMNDI SECTION:
- Must start with <dmndi:DMNDI>
- Inside: one <dmndi:DMNDiagram id=\"DMNDiagram_1\" name=\"DRD\">

SHAPE RULES:
- For each inputData:
  <dmndi:DMNShape id=\"DMNShape_<inputDataId>\" dmnElementRef=\"<inputDataId>\">
    <dc:Bounds x=\"X\" y=\"Y\" width=\"125\" height=\"45\"/>
  </dmndi:DMNShape>

- For each decision:
  <dmndi:DMNShape id=\"DMNShape_<decisionId>\" dmnElementRef=\"<decisionId>\">
    <dc:Bounds x=\"X\" y=\"Y\" width=\"180\" height=\"80\"/>
  </dmndi:DMNShape>

EDGE RULES (THIS IS THE CRITICAL FIX):
- For each informationRequirement, create ONE DMNEdge:
  <dmndi:DMNEdge id=\"DMNEdge_<informationRequirementId>\"
                 dmnElementRef=\"<informationRequirementId>\">
    <di:waypoint x=\"SOURCE_CENTER_X\" y=\"SOURCE_BOTTOM_Y\"/>
    <di:waypoint x=\"TARGET_CENTER_X\" y=\"TARGET_TOP_Y\"/>
  </dmndi:DMNEdge>

- dmnElementRef on DMNEdge MUST be the informationRequirement id
- NEVER point dmnElementRef to a decision or inputData id on an edge
- Waypoints: source = bottom-center of inputData shape, target = top-center of decision shape


(Format)
Convert the following description into DMN XML: '{description}'"""

# ── 5. Clean model output ─────────────────
def clean_model_output(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```xml\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    xml_start = text.find("<?xml")
    if xml_start == -1:
        xml_start = text.find("<definitions")

    if xml_start != -1:
        text = text[xml_start:]

    xml_end = text.rfind("</definitions>")
    if xml_end != -1:
        text = text[:xml_end + len("</definitions>")]

    return text.strip()

# ── 5b. Fix DMNEdge references ─────────────────
DMN_NS   = "https://www.omg.org/spec/DMN/20191111/MODEL/"
DMNDI_NS = "https://www.omg.org/spec/DMN/20191111/DMNDI/"
DI_NS    = "https://www.omg.org/spec/DMN/20180521/DI/"

def fix_dmn_edges(xml_string: str) -> str:
    try:
        root = etree.fromstring(xml_string.encode("utf-8"))
    except etree.XMLSyntaxError as e: # Catch specific error
        print(f"  [fix] XML parse failed: {e}")
        print(f"  [fix] Malformed XML (from fix_dmn_edges):\n{xml_string[:500]}...\n") # Print part of it
        return xml_string
    except Exception as e:
        print(f"  [fix] XML parse failed (non-syntax error): {e}")
        return xml_string

    # Collect all informationRequirement ids
    ir_map = {}  # ir_id -> (source_ref, decision_id)
    for decision in root.findall(f".//{{{DMN_NS}}}decision"):
        decision_id = decision.get("id")
        for ir in decision.findall(f"{{{DMN_NS}}}informationRequirement"):
            ir_id = ir.get("id")
            req_input = ir.find(f"{{{DMN_NS}}}requiredInput")
            req_decision = ir.find(f"{{{DMN_NS}}}requiredDecision")
            source_ref = None
            if req_input is not None:
                source_ref = req_input.get("href", "").lstrip("#")
            elif req_decision is not None:
                source_ref = req_decision.get("href", "").lstrip("#")
            if ir_id and source_ref:
                ir_map[ir_id] = (source_ref, decision_id)

    # Fix every DMNEdge whose dmnElementRef doesn't point to a valid IR
    used_ir_ids = set()
    fixed = 0
    for edge in root.findall(f".//{{{DMNDI_NS}}}DMNEdge"):
        ref = edge.get("dmnElementRef", "")
        if ref in ir_map:
            used_ir_ids.add(ref)
            continue
        # Assign the next unused IR id
        for ir_id in ir_map:
            if ir_id not in used_ir_ids:
                edge.set("dmnElementRef", ir_id)
                used_ir_ids.add(ir_id)
                fixed += 1
                break

    if fixed:
        print(f"  [fix] Patched {fixed} DMNEdge dmnElementRef(s)")

    return etree.tostring(root, pretty_print=True,
                          xml_declaration=True, encoding="UTF-8").decode("utf-8")

# ── 6. Run generation and validation ─────────────────

prompt = build_zero_shot_prompt(description)

valid_dmn_files_for_download = []

for iteration in range(1, N_ITERATIONS + 1):
    print(f"▶ Gemini zero-shot | {description_id} | temp={TEMPERATURE} | iter={iteration}")

    response = client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=TEMPERATURE,
        max_output_tokens=MAX_TOKENS,
        top_p=TOP_P,
    ),
)

    usage = response.usage_metadata

    print("Input tokens:", usage.prompt_token_count)
    print("Output tokens:", usage.candidates_token_count)
    print("Total tokens:", usage.total_token_count)

    raw_output = response.text

    dmn_xml = clean_model_output(raw_output)
    dmn_xml = fix_dmn_edges(dmn_xml)

    base_name = f"{description_id}_gemini_zero_shot_temp_{TEMPERATURE}_iter_{iteration}"
    dmn_path = f"experiments/dmn/{base_name}.dmn"

    with open(dmn_path, "w", encoding="utf-8") as f:
        f.write(dmn_xml)

    print(f"Saved: {dmn_path}")

    is_valid, errors = validate_dmn_xml(dmn_xml)

    if is_valid:
        print("✅ DMN XML passed available validation.")
        if errors:
            for warning in errors:
                print(f"  - {warning}")
    else:
        print("❌ DMN XML is INVALID. Errors:")
        for error in errors:
            print(f"  - {error}")

# ── 7. Download DMN results (only valid ones) ─────────────────
    files.download(dmn_path)

DMN schema parsed successfully.
▶ Gemini zero-shot | description_1 | temp=0.2 | iter=1
Input tokens: 1961
Output tokens: 1578
Total tokens: 11492
Saved: experiments/dmn/description_1_gemini_zero_shot_temp_0.2_iter_1.dmn
✅ DMN XML passed available validation.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

▶ Gemini zero-shot | description_1 | temp=0.2 | iter=2
Input tokens: 1961
Output tokens: 1819
Total tokens: 31410
Saved: experiments/dmn/description_1_gemini_zero_shot_temp_0.2_iter_2.dmn
✅ DMN XML passed available validation.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

▶ Gemini zero-shot | description_1 | temp=0.2 | iter=3
Input tokens: 1961
Output tokens: 1399
Total tokens: 36957
  [fix] XML parse failed: AttValue: ' expected, line 121, column 71 (<string>, line 121)
  [fix] Malformed XML (from fix_dmn_edges):
<?xml version="1.0" encoding="UTF-8"?>
<definitions id="Definitions_00ghp5h" name="DRD" namespace="http://example.com/dmn" exporter="Camunda Modeler" exporterVersion="5.44.0" modeler:executionPlatform="Camunda Cloud" modeler:executionPlatformVersion="8.8.0" xmlns="https://www.omg.org/spec/DMN/20191111/MODEL/" xmlns:dmndi="https://www.omg.org/spec/DMN/20191111/DMNDI/" xmlns:dc="http://www.omg.org/spec/DMN/20180521/DC/" xmlns:modeler="http://camunda.org/schema/modeler/1.0" xmlns:biodi="http://bpmn...

Saved: experiments/dmn/description_1_gemini_zero_shot_temp_0.2_iter_3.dmn
  [validation] XML Syntax Error: AttValue: ' expected, line 121, column 71 (<string>, line 121)
  [validation] Malformed XML (from validate_dmn_xml):
<?xml version="1.0" enco

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>